In [1]:
pip install PyPDF2 scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import PyPDF2
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
def read_resume(file_path):
    text = ""

    try:
        file = open(file_path, "rb")
        reader = PyPDF2.PdfReader(file)

        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text

        file.close()

    except Exception as e:
        print("Error reading resume:", e)

    return text
    

In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [5]:
skill_list = [
    "python", "java", "c++", "sql",
    "machine learning", "data science",
    "pandas", "numpy", "flask", "django",
    "deep learning", "nlp", "statistics",
    "html", "css", "javascript"
]

In [6]:
def extract_skills(text):
    found = []
    missing = []

    for skill in skill_list:
        if skill in text:
            found.append(skill)
        else:
            missing.append(skill)

    return found, missing

In [7]:
def match_resume(resume_text, job_text):
    docs = [resume_text, job_text]

    tfidf = TfidfVectorizer()
    matrix = tfidf.fit_transform(docs)

    score = cosine_similarity(matrix[0], matrix[1])[0][0]
    return score * 100

In [8]:
def ats_score(match_score, skills_found):
    skill_ratio = len(skills_found) / len(skill_list)

    final_score = (0.6 * (match_score / 100)) + (0.4 * skill_ratio)
    return round(final_score * 100, 2)

In [9]:
def get_recommendation(score):
    if score >= 75:
        return "Strong match for this role"
    elif score >= 50:
        return "Moderate match, improve skills"
    else:
        return "Weak match, needs strong improvement"

In [10]:
def generate_report(resume_text, job_text):

    print("\n================ RESUME ANALYZER REPORT ================\n")

    # clean data
    resume_text = clean_text(resume_text)
    job_text = clean_text(job_text)

    # score calculation
    match = match_resume(resume_text, job_text)

    # skill extraction
    found, missing = extract_skills(resume_text)

    # final ATS score
    final = ats_score(match, found)

    print("MATCH SCORE:", round(match, 2), "%")
    print("ATS SCORE:", final, "%")

    print("\n--- SKILLS FOUND ---")
    for s in found:
        print("-", s)

    print("\n--- SKILLS MISSING ---")
    for s in missing[:12]:
        print("-", s)

    print("\n--- RECOMMENDATION ---")
    print(get_recommendation(final))

    print("\n========================================================\n")

In [13]:
resume_text = read_resume("C:/Users/786/DATA ANALYTICS/r/abd_test_resume.pdf")

job_description = """
We are hiring a python developer with machine learning, sql, pandas and flask.
"""

generate_report(resume_text, job_description)


================ RESUME ANALYZER REPORT ================

MATCH SCORE: 5.84 %
ATS SCORE: 6.0 %

--- SKILLS FOUND ---
- python

--- SKILLS MISSING ---
- java
- c++
- sql
- machine learning
- data science
- pandas
- numpy
- flask
- django
- deep learning
- nlp
- statistics

--- RECOMMENDATION ---
Weak match, needs strong improvement


